In [1]:
import os

import torch
import sqlite3
import pandas as pd

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH

EMBED_PATH = EMBEDS_DIR / "dino/20260801_221150"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "ellipsoid_bootsrap"
RESULTS_DIR = RESULTS / EMBED_NAME

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [5]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid.factories import create_cover, create_evaluator
from src.algorithims.ellipsoid.bootstrap import BootstrapRunner
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

In [6]:
test_df : list[pd.DataFrame] = []
train_df: list[pd.DataFrame] = []

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category 
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

   
    runner = BootstrapRunner(
        cover_factory=create_cover,
        eval_factory=create_evaluator,
        n_test_bootstraps=1000,
        n_train_bootstraps=100,
        seed=42,
        max_workers=2
    )

    test_bootstraps, train_bootstaps = runner.run(
        train_emb=cat_emb, 
        good_test_emb=good_test_emb, 
        defect_test_emb=defect_test_emb
        )

    test_df.append(test_bootstraps)
    train_df.append(train_bootstaps)

Running bottle


Test Bootstraps:   0%|          | 0/1000 [00:00<?, ?it/s]

Worker StartedWorker Started

Cover CreatedCover Created

Evaluator CreatedEvaluator Created



Train Bootstrap [0-49]: 0it [00:00, ?it/s]

Train Bootstrap [50-99]: 0it [00:00, ?it/s]

Starting Bootstrap
Starting Bootstrap
Samples Created
Samples CreatedEmbeddings copied

Embeddings copied


KeyboardInterrupt: 